# 🚀 Enterprise Banking Fraud & Risk Intelligence Platform
## Cloud Training Pipeline: LightGBM + GAT Graph + Autoencoder + SMOTE Ablation
**Author:** Karthikeya Challa (23WU0104013)  
**Dataset:** PaySim Mobile Banking & Fund Transfers (6.3M records, Lopez-Rojas et al.)  
**Environment:** Kaggle Cloud GPU / High-RAM CPU

### Pipeline Highlights:
1. Ingestion of 6.3M mobile money transfers (TRANSFER, CASH_OUT)
2. Banking feature engineering (balance drainage, error deltas, step hours)
3. **Empirical Ablation Study:** Baseline vs. 2002 SMOTE vs. Modern Cost-Sensitive Learning
4. 5-Fold Stratified Cross-Validation on LightGBM
5. Semi-supervised Deep Autoencoder Anomaly Detection
6. PyTorch Geometric Bipartite Graph Attention Network (GAT) for Money Mule Detection
7. TreeSHAP feature attributions and artifact export to `/kaggle/working/models`

### Cell 1: Environment Setup & Package Verification

In [ ]:
import os, sys, gc, time, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import lightgbm as lgb
import shap
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Training Device: {DEVICE}')
print(f'✅ LightGBM Version: {lgb.__version__}')
print(f'✅ PyTorch Version: {torch.__version__}')

OUTPUT_DIR = '/kaggle/working/models'
os.makedirs(OUTPUT_DIR, exist_ok=True)


### Cell 2: Ingest PaySim Mobile Banking Dataset

In [ ]:
input_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.csv'):
            input_files.append(os.path.join(root, file))

print('Available input CSVs:', input_files)
csv_path = [f for f in input_files if 'paysim' in f.lower() or 'fraud' in f.lower()][0]
print(f'Loading dataset from: {csv_path}')

df_raw = pd.read_csv(csv_path)
print(f'Total raw transactions: {len(df_raw):,}')

df = df_raw[df_raw['type'].isin(['TRANSFER', 'CASH_OUT'])].copy().reset_index(drop=True)
print(f'Filtered to high-risk transfer channels: {len(df):,} transactions')
print(f'Fraud rate: {df["isFraud"].mean()*100:.2f}% ({df["isFraud"].sum():,} fraud txns)')
del df_raw; gc.collect()


### Cell 3: Feature Engineering for Digital Banking Transfers

In [ ]:
df['hour_of_day'] = df['step'] % 24
df['is_night'] = ((df['hour_of_day'] < 5) | (df['hour_of_day'] >= 23)).astype(int)

df['error_orig'] = (df['oldbalanceOrg'] - df['newbalanceOrig']) - df['amount']
df['error_dest'] = (df['newbalanceDest'] - df['oldbalanceDest']) - df['amount']
df['drain_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1.0)
df['amt_log'] = np.log1p(df['amount'])

df['is_transfer'] = (df['type'] == 'TRANSFER').astype(int)
df['is_cash_out'] = (df['type'] == 'CASH_OUT').astype(int)

feature_cols = [
    'amount', 'amt_log', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'error_orig', 'error_dest',
    'drain_ratio', 'hour_of_day', 'is_night', 'is_transfer', 'is_cash_out'
]

X = df[feature_cols].copy()
y = df['isFraud'].values
print(f'Engineered features: {len(feature_cols)}')


### Cell 4: SMOTE vs. Cost-Sensitive Ablation Study (Addressing Professor\'s Question)

In [ ]:
print('='*70)
print('  RUNNING ABLATION STUDY: BASELINE vs SMOTE vs COST-SENSITIVE  ')
print('='*70)

X_sample, _, y_sample, _ = train_test_split(X, y, train_size=200000, stratify=y, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_sample, y_sample, test_size=0.25, stratify=y_sample, random_state=42)

ablation_results = []

# 1. Baseline
clf_base = lgb.LGBMClassifier(n_estimators=250, learning_rate=0.08, random_state=42, n_jobs=-1, verbose=-1)
clf_base.fit(X_tr, y_tr)
p_base = clf_base.predict_proba(X_val)[:, 1]
b_base = (p_base >= 0.5).astype(int)
ablation_results.append({
    'Strategy': 'Baseline (Raw Unbalanced)',
    'AUC-ROC': roc_auc_score(y_val, p_base),
    'Precision': precision_score(y_val, b_base, zero_division=0),
    'Recall': recall_score(y_val, b_base, zero_division=0),
    'F1-Score': f1_score(y_val, b_base, zero_division=0)
})

# 2. SMOTE
sm = SMOTE(random_state=42, k_neighbors=5)
X_tr_sm, y_tr_sm = sm.fit_resample(X_tr, y_tr)
clf_smote = lgb.LGBMClassifier(n_estimators=250, learning_rate=0.08, random_state=42, n_jobs=-1, verbose=-1)
clf_smote.fit(X_tr_sm, y_tr_sm)
p_smote = clf_smote.predict_proba(X_val)[:, 1]
b_smote = (p_smote >= 0.5).astype(int)
ablation_results.append({
    'Strategy': 'SMOTE (2002 Synthetic)',
    'AUC-ROC': roc_auc_score(y_val, p_smote),
    'Precision': precision_score(y_val, b_smote, zero_division=0),
    'Recall': recall_score(y_val, b_smote, zero_division=0),
    'F1-Score': f1_score(y_val, b_smote, zero_division=0)
})

# 3. Cost-Sensitive Learning
scale_w = (y_tr == 0).sum() / max(1, (y_tr == 1).sum())
clf_cost = lgb.LGBMClassifier(n_estimators=250, learning_rate=0.08, scale_pos_weight=scale_w, random_state=42, n_jobs=-1, verbose=-1)
clf_cost.fit(X_tr, y_tr)
p_cost = clf_cost.predict_proba(X_val)[:, 1]
b_cost = (p_cost >= 0.5).astype(int)
ablation_results.append({
    'Strategy': 'Cost-Sensitive (Asymmetric Loss)',
    'AUC-ROC': roc_auc_score(y_val, p_cost),
    'Precision': precision_score(y_val, b_cost, zero_division=0),
    'Recall': recall_score(y_val, b_cost, zero_division=0),
    'F1-Score': f1_score(y_val, b_cost, zero_division=0)
})

ablation_df = pd.DataFrame(ablation_results)
print(ablation_df.to_string(index=False))
ablation_df.to_csv(f'{OUTPUT_DIR}/ablation_study_results.csv', index=False)


### Cell 5: Train Production Cost-Sensitive LightGBM (5-Fold CV)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
with open(f'{OUTPUT_DIR}/scaler.pkl', 'wb') as f: pickle.dump(scaler, f)
with open(f'{OUTPUT_DIR}/feature_columns.pkl', 'wb') as f: pickle.dump(feature_cols, f)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
prod_model = None
pos_ratio = (y == 0).sum() / max(1, (y == 1).sum())

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_f_tr, y_f_tr = X.iloc[trn_idx], y[trn_idx]
    X_f_va, y_f_va = X.iloc[val_idx], y[val_idx]
    m = lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.06, num_leaves=63,
        scale_pos_weight=pos_ratio, random_state=42, n_jobs=-1, verbose=-1
    )
    m.fit(X_f_tr, y_f_tr, eval_set=[(X_f_va, y_f_va)], callbacks=[lgb.early_stopping(40, verbose=False)])
    p_val = m.predict_proba(X_f_va)[:, 1]
    oof_preds[val_idx] = p_val
    print(f'   Fold {fold}/5 — AUC: {roc_auc_score(y_f_va, p_val):.4f}')
    prod_model = m

final_auc = roc_auc_score(y, oof_preds)
print(f'\n✅ Final Out-Of-Fold AUC: {final_auc:.4f}')
with open(f'{OUTPUT_DIR}/lightgbm_model.pkl', 'wb') as f:
    pickle.dump(prod_model, f)
print(f'💾 LightGBM saved to {OUTPUT_DIR}/lightgbm_model.pkl')


### Cell 6: Semi-Supervised Autoencoder for Zero-Day Anomaly Detection

In [ ]:
class BankingAutoencoder(nn.Module):
    def __init__(self, in_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, in_dim)
        )
    def forward(self, x): return self.decoder(self.encoder(x))
    def get_mse(self, x):
        with torch.no_grad():
            return torch.mean((x - self.forward(x))**2, dim=1)

X_legit = X_scaled[y == 0][:120000]
ae_loader = DataLoader(TensorDataset(torch.FloatTensor(X_legit)), batch_size=2048, shuffle=True)
ae_model = BankingAutoencoder(in_dim=X_scaled.shape[1], latent_dim=16).to(DEVICE)
ae_opt = optim.Adam(ae_model.parameters(), lr=1e-3)
ae_crit = nn.MSELoss()

for ep in range(1, 21):
    ae_model.train()
    l_sum = 0
    for (batch,) in ae_loader:
        b = batch.to(DEVICE)
        ae_opt.zero_grad()
        loss = ae_crit(ae_model(b), b)
        loss.backward()
        ae_opt.step()
        l_sum += loss.item()
    if ep % 5 == 0 or ep == 1:
        print(f'   Autoencoder Epoch {ep:02d}/20 — MSE Loss: {l_sum/len(ae_loader):.6f}')

torch.save(ae_model.state_dict(), f'{OUTPUT_DIR}/autoencoder.pt')
print(f'💾 Autoencoder saved to {OUTPUT_DIR}/autoencoder.pt')


### Cell 7: Bipartite Graph Attention Network (GAT) for Money Mule Detection

In [ ]:
class GATLayer(nn.Module):
    def __init__(self, in_f, out_f, alpha=0.2):
        super().__init__()
        self.W = nn.Parameter(torch.empty(size=(in_f, out_f)))
        nn.init.xavier_uniform_(self.W.data, gain=1.414)
        self.a = nn.Parameter(torch.empty(size=(2 * out_f, 1)))
        nn.init.xavier_uniform_(self.a.data, gain=1.414)
        self.leakyrelu = nn.LeakyReLU(alpha)
    def forward(self, h, adj):
        Wh = torch.mm(h, self.W)
        N = Wh.size(0)
        a_input = torch.cat([Wh.repeat(1, N).view(N * N, -1), Wh.repeat(N, 1)], dim=1).view(N, N, -1)
        e = self.leakyrelu(torch.matmul(a_input, self.a).squeeze(2))
        attention = F.softmax(torch.where(adj > 0, e, -9e15 * torch.ones_like(e)), dim=1)
        return F.elu(torch.matmul(attention, Wh))

class TransferGAT(nn.Module):
    def __init__(self, in_f, nhid=32, nout=16):
        super().__init__()
        self.gat1 = GATLayer(in_f, nhid)
        self.gat2 = GATLayer(nhid, nout)
        self.out = nn.Linear(nout, 1)
    def forward(self, x, adj):
        h = self.gat2(self.gat1(x, adj), adj)
        return torch.sigmoid(self.out(h)), h

sub_df = df.sample(n=2500, random_state=42).reset_index(drop=True)
X_sub = scaler.transform(sub_df[feature_cols])
y_sub = sub_df['isFraud'].values
N = len(sub_df)
adj = np.eye(N, dtype=np.float32)
dests = sub_df['nameDest'].values
origs = sub_df['nameOrig'].values

edges = 0
for i in range(N):
    for j in range(i + 1, min(i + 200, N)):
        if dests[i] == dests[j] or origs[i] == origs[j]:
            adj[i, j] = 1.0; adj[j, i] = 1.0; edges += 2

print(f'✅ Relational Graph: {N} Nodes, {edges} Mule Edges.')
gat = TransferGAT(in_f=X_sub.shape[1], nhid=32, nout=16).to(DEVICE)
g_opt = optim.Adam(gat.parameters(), lr=0.01)
x_t = torch.FloatTensor(X_sub).to(DEVICE)
adj_t = torch.FloatTensor(adj).to(DEVICE)
y_t = torch.FloatTensor(y_sub).unsqueeze(1).to(DEVICE)
crit = nn.BCELoss()

for ep in range(1, 26):
    gat.train()
    g_opt.zero_grad()
    preds, _ = gat(x_t, adj_t)
    loss = crit(preds, y_t)
    loss.backward()
    g_opt.step()
    if ep % 5 == 0:
        gat_auc = roc_auc_score(y_sub, preds.detach().cpu().numpy().flatten())
        print(f'   GAT Epoch {ep:02d}/25 — Loss: {loss.item():.4f} | Relational Graph AUC: {gat_auc:.4f}')

torch.save(gat.state_dict(), f'{OUTPUT_DIR}/gat_model.pt')
print(f'💾 GAT Model saved to {OUTPUT_DIR}/gat_model.pt')


### Cell 8: TreeSHAP Feature Attributions & Model Export

In [ ]:
print('Generating TreeSHAP Explainer...')
explainer = shap.TreeExplainer(prod_model)
with open(f'{OUTPUT_DIR}/shap_explainer.pkl', 'wb') as f:
    pickle.dump(explainer, f)

print('='*70)
print('  🎉 ALL KAGGLE TRAINING FINISHED SUCCESSFULLY!')
print('='*70)
for f in os.listdir(OUTPUT_DIR):
    sz = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
    print(f'   • {f:<30} ({sz:.1f} KB)')
